# Split Tumor Audit — Zero-Tumor & Low-Burden Volume Analysis
**Goal:** Confirm or rule out whether val/test zero-tumor and low-burden volumes are unevenly distributed before trusting AUPRC numbers from the sweep.

---
## 1. Setup

In [ ]:
import sys, os, warnings
from pathlib import Path
# pyarrow TxF workaround (must be set before any sklearn import)
os.environ["PYARROW_IGNORE_ZERO_COPY"] = "1"
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

sys.path.insert(0, str(Path.cwd().parent))
from src.data.split_audit import (
    load_volume_manifest,
    flag_zero_tumor_volumes,
    split_composition_table,
    plot_tumor_burden_by_split,
    save_audit_outputs,
)

print("Setup OK")

---
## 2. Load Volume Manifest

In [ ]:
IMAGES_DIR = r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver Img Dataset"
MASKS_DIR = r"D:\DATA SCIENCE AND ANALYTICS\Dataset\LiTS_masks"
SPLITS_DIR = Path.cwd().parent / "data" / "splits"

df = load_volume_manifest(IMAGES_DIR, MASKS_DIR, str(SPLITS_DIR))
df = flag_zero_tumor_volumes(df, burden_low_threshold=0.05)
print(f"Loaded {len(df)} volumes")
df.head()

---
## 3. Split Composition Table

In [ ]:
comp = split_composition_table(df)
comp

---
## 4. Zero-Tumor Volume Distribution Across Splits

In [ ]:
# Are any of the zero-tumor volumes concentrated in val or test?
zero_summary = df[df["is_zero_tumor"]].groupby("split").agg(
    n_zero_tumor_volumes=("volume_id", "count"),
    volume_ids=("volume_id", list),
).reset_index()
zero_summary

In [ ]:
# Also check low-burden (<0.05%) volumes
low_summary = df[df["is_low_burden"]].groupby("split").agg(
    n_low_burden_volumes=("volume_id", "count"),
    volume_ids=("volume_id", list),
).reset_index()
low_summary

---
## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 5a. Zero-tumor volume counts per split
zero_counts = df.groupby("split")["is_zero_tumor"].sum()
colors = ["#2ecc71", "#3498db", "#e74c3c"]
for i, s in enumerate(["train", "val", "test"]):
    if s in zero_counts.index:
        axes[0].bar(i, zero_counts[s], color=colors[i], alpha=0.8, edgecolor="black")
        axes[0].text(i, zero_counts[s] + 0.2, str(int(zero_counts[s])), ha="center", fontweight="bold")
axes[0].set_xticks([0, 1, 2]); axes[0].set_xticklabels(["Train", "Val", "Test"])
axes[0].set_ylabel("Zero-Tumor Volumes"); axes[0].set_title("Zero-Tumor Volume Count by Split")
axes[0].grid(True, axis="y")

# 5b. Low-burden (<0.05%) volume counts per split
low_counts = df.groupby("split")["is_low_burden"].sum()
for i, s in enumerate(["train", "val", "test"]):
    if s in low_counts.index:
        axes[1].bar(i, low_counts[s], color=colors[i], alpha=0.8, edgecolor="black")
        axes[1].text(i, low_counts[s] + 0.2, str(int(low_counts[s])), ha="center", fontweight="bold")
axes[1].set_xticks([0, 1, 2]); axes[1].set_xticklabels(["Train", "Val", "Test"])
axes[1].set_ylabel("Low-Burden Volumes (<0.05%)"); axes[1].set_title("Low-Burden Volume Count by Split")
axes[1].grid(True, axis="y")

# 5c. Volume distribution across splits (pie)
vol_counts = df["split"].value_counts()
axes[2].pie(
    [vol_counts.get(s, 0) for s in ["train", "val", "test"]],
    labels=["Train", "Val", "Test"],
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
    explode=(0, 0.05, 0.05),
)
axes[2].set_title("Volume Distribution Across Splits")

plt.tight_layout()
plt.savefig(Path("../experiments/sprint1") / "split_audit_barplots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5d. Faceted histogram of tumor burden per split
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for i, s in enumerate(["train", "val", "test"]):
    subset = df[df["split"] == s]["tumor_burden_pct"]
    axes[i].hist(subset, bins=30, color=colors[i], alpha=0.7, edgecolor="black", linewidth=0.5)
    axes[i].axvline(subset.median(), color="black", ls="--", lw=2, label=f"Median={subset.median():.4f}%")
    axes[i].set_xlabel("Tumor Burden (%)")
    axes[i].set_title(f"{'Train' if s == 'train' else s.upper()}")
    axes[i].legend(fontsize=9)
axes[0].set_ylabel("Count")
plt.suptitle("Tumor Burden Distribution per Split", fontsize=14)
plt.tight_layout()
plt.savefig(Path("../experiments/sprint1") / "split_audit_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Save Audit Outputs

In [ ]:
import os
OUTPUT_DIR = "../experiments/sprint1"
os.makedirs(Path(OUTPUT_DIR).resolve(), exist_ok=True)
save_audit_outputs(df, comp, output_dir=OUTPUT_DIR)
print("Audit outputs saved.")

---
## 7. Tumor Burden Distribution by Split

In [ ]:
plot_tumor_burden_by_split(df)

---
## 8. Per-Volume Burden Detail (Low-Burden Volumes)

In [ ]:
low_vols = df[df["tumor_burden_pct"] < 0.05].sort_values("tumor_burden_pct", ascending=False)
print(f"Volumes with burden < 0.05%: {len(low_vols)}")
low_vols[["volume_id", "split", "n_slices", "tumor_positive_slices", "tumor_burden_pct"]]

---
## 9. Decision Cell

**Assessment:** Are val/test splits structurally skewed by zero-tumor or low-burden volumes?

Fill in after reviewing tables above:
- Zero-tumor volumes: ___ in train, ___ in val, ___ in test
- Low-burden (<0.05%) volumes: ___ in train, ___ in val, ___ in test
- Any split disproportionately affected? ___

**Conclusion:** ___ (acceptable / needs re-splitting before sweep)

In [ ]:
# Print the raw numbers for the decision cell above
for split_name in ["train", "val", "test"]:
    subset = df[df["split"] == split_name]
    n_zero = subset["is_zero_tumor"].sum()
    n_low = subset["is_low_burden"].sum()
    print(f"{split_name.upper():5s}: {len(subset):3d} vols, {int(n_zero):2d} zero-tumor, {int(n_low):2d} low-burden")